In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)

In [ ]:
from read_data.read_data import read_data

df = read_data(50000)
df.head(10)

In [ ]:
from read_data.translate_names import translate_to_english

df = translate_to_english(df)
df.head(10)

In [ ]:
from read_data.fix_types import fix_data_types

df = fix_data_types(df)
df.head(10)

In [ ]:
from data_cleaning.feature_engineering import identify_unemployment_episodes

df = identify_unemployment_episodes(df)
df.head(10)

In [ ]:
from data_cleaning.feature_engineering import generate_biographical_variables

df = generate_biographical_variables(df)
df.head(10)

In [ ]:
from data_cleaning.feature_engineering import index_unemployment_entries

df = index_unemployment_entries(df)
df.head(10)

In [ ]:
from data_cleaning.feature_engineering import generate_employment_history

df = generate_employment_history(df, 1)
df.head(10)

In [ ]:
from data_cleaning.transform import reshape_data_for_ml

df = reshape_data_for_ml(df)
df.head(10)

In [ ]:
# Nasty imputation workaround to get things running
# TODO: REMOVE THIS!

# Replace anything that is NA with a 0

df = df.replace(99999, pd.NA)
df = df.fillna(0)
df.head()

## Modeling

With the data prepared, we can now move on to modeling. This involves selecting features and target variables, splitting the data into training and testing sets, and training a machine learning model.

In [ ]:
from modeling.setup import select_features

X = select_features(df)
X.head(10)

In [ ]:
X.columns

In [ ]:
from modeling.setup import select_target

y = select_target(df)
y.head(10)

In [ ]:
print(f"Feature Shape: {X.shape}")
print(f"Target Shape: {y.shape}")
print(f"Target distribution (full): {y.value_counts(normalize=True)}")

In [ ]:
from modeling.data_split import split_data

X_train, X_test, y_train, y_test = split_data(X, y)

In [ ]:
print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Target distribution (train): {y_train.value_counts(normalize=True)}")

In [ ]:
from sklearn.linear_model import LogisticRegression

# Train logistic regression model
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight="balanced",  # Handle class imbalance
    solver="lbfgs",
)

print("Training logistic regression model...")
lr_model.fit(X_train, y_train)
print("Model trained successfully!")

print(f"\nTraining Accuracy: {lr_model.score(X_train, y_train):.4f}")
print(f"Test Accuracy: {lr_model.score(X_test, y_test):.4f}")

In [ ]:
# Make predictions
y_pred_train = lr_model.predict(X_train)
y_pred_test = lr_model.predict(X_test)

# Get prediction probabilities
y_pred_proba_train = lr_model.predict_proba(X_train)[:, 1]
y_pred_proba_test = lr_model.predict_proba(X_test)[:, 1]

In [ ]:
from modeling.evaluation import evaluate_predictions

# Evaluate on training set
print("TRAINING SET")
train_results = evaluate_predictions(y_train, y_pred_train, y_pred_proba_train)

In [ ]:
# Evaluate on test set
print("TEST SET")
test_results = evaluate_predictions(y_test, y_pred_test, y_pred_proba_test)